#### EDA 01: Orders table
**File:** `olist_orders_dataset.csv`  |  **Goal:** understand the orders table

Orders table EDA: key pointers with explanation

1. Grain

- Checked: whether order_id is unique and whether any rows are repeated.
- Found: 99,441 rows, all order_id values unique, no duplicated rows. customer_id is also unique, but only because a new one is created for every order.
- Why it matters: the other tables (items, payments, reviews) have several rows per order. Knowing that one row is one order tells us they must be collapsed to order level before joining. Otherwise the row count silently grows.
- Decision: the grain is one row per order. A real person is customer_unique_id, which sits in the customers table.

2. Status

- Checked: the final state of each order.
- Found: 97.0% are delivered (96,478). The other 2,963 are shipped, canceled, unavailable, invoiced, processing, created or approved.
- Why it matters: the data is strongly imbalanced. Delivered is the normal case and the rest are exceptions that need their own explanation.
- Decision: study the exceptions instead of dropping them.

3. Missing values

- Checked: missing count and percentage for every column.
- Found: only 3 of 8 columns have gaps: approval time (160, 0.16%), carrier date (1,783, 1.79%) and delivery date (2,965, 2.98%). 96,461 orders have all dates, and the gaps follow the order lifecycle (purchase, approval, handoff, delivery).
- Why it matters: gaps that follow a pattern are not random. They mean something.
- Decision: do not fill them with a mean or a default.

4. Two kinds of missing

- The event never happened: unavailable, invoiced, processing, shipped and canceled orders. The empty date is the information. Filling it would invent events.
- The event happened but was not recorded: a few delivered orders (14 without approval time, 2 without carrier date, 8 without delivery date). These are recording gaps.

5. Contradictions

- Checked: whether status and dates agree.
- Found: 8 delivered orders have no delivery date, and 6 canceled orders have a full set of dates including delivery. 


6. Time

- Checked: orders per month, weekday and time of day.
- Found: volume grew from 800 orders in Jan 2017 to a peak of 7,544 in Nov 2017, then stayed near 6,000 to 7,300 per month until Aug 2018. 2016 has only 329 orders and Sep and Oct 2018 only 20. The last delivered order was bought on 2018-08-29. Mondays are busiest and Saturdays quietest.
- Why it matters: the start is thin and the end is unfinished, because late orders cannot show a delivery yet.
Decision: propose a modelling window of about Jan 2017 to Aug 2018 (final decision in Part 2).

7. Date logic

- Checked: whether dates follow the real order of events.
- Found: no order was approved, delivered or estimated before purchase. But 1,359 orders have the carrier date before approval (166 of them even before purchase), and 23 were delivered before the carrier handoff.
- Why it matters: timestamps are not fully reliable, probably because they were recorded late or wrongly.

8. Durations and outliers

- Checked: how long each step takes.
- Found: approval has a median of 0.34 hours but a maximum of 4,509 hours. Purchase to delivery has a median of 10.2 days and a maximum of 209.6 days. The promised time has a median of 23.2 days, more than double the real time, so the promise is padded.
Why it matters: all durations have a long right tail. These are real slow orders, not errors.
- Decision: keep outliers. Cap or log values only inside the model pipeline.

#### Load and first look

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

RAW = Path("../../data/raw")                      # notebook runs from the notebooks folder
orders_raw = pd.read_csv(RAW / "olist_orders_dataset.csv")
orders = orders_raw.copy()                     # all changes happen on the copy, raw stays untouched

print("shape:", orders.shape)
orders.head(3)

shape: (99441, 8)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


- The table has 99,441 rows and 8 columns. 
- Three columns are text (two ids and the status) and five are dates. 
- The first rows show the normal journey of an order: purchase, approval, handoff to the carrier, delivery, and the estimated delivery date promised to the customer.

#### Data types and date range

In [2]:
# Pick date columns by name ending, so none is missed
DATE_COLS = [c for c in orders.columns if c.endswith(("_timestamp", "_date", "_at"))]

for c in DATE_COLS:
    # No errors="coerce": a bad date should fail loudly, not turn into NaT silently
    orders[c] = pd.to_datetime(orders[c])

orders[DATE_COLS].agg(["min", "max"]).T

,min,max
order_purchase_timestamp,2016-09-04 21:15:19,2018-10-17 17:30:18
order_approved_at,2016-09-15 12:16:38,2018-09-03 17:40:06
order_delivered_carrier_date,2016-10-08 10:34:01,2018-09-11 19:48:28
order_delivered_customer_date,2016-10-11 13:46:32,2018-10-17 13:22:46
order_estimated_delivery_date,2016-09-30 00:00:00,2018-11-12 00:00:00


- All five date columns convert without errors. 
- Purchases run from 2016-09-04 to 2018-10-17, about two years. 

#### Keys and duplicates

In [3]:
print("order_id unique          :", orders["order_id"].is_unique)
print("customer_id unique       :", orders["customer_id"].is_unique)
print("fully duplicated rows    :", orders.duplicated().sum())
# Two orders in the same second are possible. That is not a duplicate.
print("repeated purchase times  :", orders["order_purchase_timestamp"].duplicated().sum())

order_id unique          : True
customer_id unique       : True
fully duplicated rows    : 0
repeated purchase times  : 566


- `order_id` is unique and there are no duplicated rows, so **one row is one order**. 
- `customer_id` is also unique, but only because a new `customer_id` is created for every order. 
- A real person is identified by `customer_unique_id` in the customers table. 
- The 566 repeated purchase times are different orders placed in the same second.

#### Order status

In [4]:
status = orders["order_status"].value_counts()
pd.DataFrame({"n_orders": status, "pct": (status / len(orders) * 100).round(2)})

,n_orders,pct
order_status,,
delivered,96478,97.02
shipped,1107,1.11
canceled,625,0.63
unavailable,609,0.61
invoiced,314,0.32
processing,301,0.30
created,5,0.01
approved,2,0.00


- 97.0% of orders are `delivered` (96,478). 
- The other 2,963 orders never reached the delivered state: 
    - shipped (1,107)
    - canceled (625)
    - unavailable (609)
    - invoiced (314)
    - processing (301)
    - created (5) 
    - approved (2). 

The Delivered data is the normal case and the rest are exceptions worth studying.

#### Missing values
#### How much is missing, and in which pattern

In [5]:
# Column-wise missing count and percentage, for every column (zeros included)
missing = pd.DataFrame({"n_rows": len(orders),
                        "n_missing": orders.isna().sum(),
                        "pct_missing": (orders.isna().mean() * 100).round(2)})
display(missing.sort_values("n_missing", ascending=False))

,n_rows,n_missing,pct_missing
order_delivered_customer_date,99441,2965,2.98
order_delivered_carrier_date,99441,1783,1.79
order_approved_at,99441,160,0.16
order_id,99441,0,0.00
order_purchase_timestamp,99441,0,0.00
order_status,99441,0,0.00
customer_id,99441,0,0.00
order_estimated_delivery_date,99441,0,0.00


- Only three columns have gaps: approval time (160), carrier date (1,783) and delivery date (2,965). 

In [6]:
LIFECYCLE_COLS = ["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]

# Every combination of missing / present across the three lifecycle dates
flags = orders[LIFECYCLE_COLS].isna()
flags.columns = ["no_approved", "no_carrier", "no_delivered"]
patterns = flags.value_counts().rename("n_orders").reset_index()
patterns["pct"] = (patterns["n_orders"] / len(orders) * 100).round(2)
patterns

,no_approved,no_carrier,no_delivered,n_orders,pct
0,False,False,False,96461,97.00
1,False,True,True,1636,1.65
2,False,False,True,1183,1.19
3,True,True,True,146,0.15
4,True,False,False,14,0.01
5,False,True,False,1,0.00


- The pattern table shows why gaps are not random: 96,461 orders (97.0%) have every date
- 1,636 miss carrier and delivery
- 1,183 miss only delivery 
- 146 miss all three dates. 
- The gaps follow the order lifecycle (purchase, approval, handoff, delivery). 

#### Missing dates by status

In [7]:
by_status = orders[LIFECYCLE_COLS].isna().groupby(orders["order_status"]).sum()
by_status["n_orders"] = orders["order_status"].value_counts()
by_status

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,n_orders
order_status,,,,
approved,0,2,2,2
canceled,141,550,619,625
created,5,5,5,5
delivered,14,2,8,96478
invoiced,0,314,314,314
processing,0,301,301,301
shipped,0,0,1107,1107
unavailable,0,609,609,609


The table explains most of the gaps. 
- `unavailable`, `invoiced` and `processing` orders never got a carrier or delivery date. 
- `shipped` orders have everything except the delivery date. 
- `canceled` orders stop at different stages (141 never approved, 550 never handed to the carrier). In all these rows the event never happened, so the empty value is the information and must not be filled.

- The `delivered` row is different. 14 delivered orders miss the approval time, 2 miss the carrier date and 8 miss the delivery date, although the status says delivered. These are recording gaps, not events that never happened. They are also the reason why 2,965 missing delivery dates do not match 2,963 non-delivered orders.

#### Status vs dates: contradictions

In [8]:
is_delivered = orders["order_status"] == "delivered"
is_canceled = orders["order_status"] == "canceled"

delivered_no_approval = is_delivered & orders["order_approved_at"].isna()
delivered_no_carrier = is_delivered & orders["order_delivered_carrier_date"].isna()
delivered_no_delivery = is_delivered & orders["order_delivered_customer_date"].isna()
canceled_has_delivery = is_canceled & orders["order_delivered_customer_date"].notna()

print("delivered, no approval time :", delivered_no_approval.sum())
print("delivered, no carrier date  :", delivered_no_carrier.sum())
print("delivered, no delivery date :", delivered_no_delivery.sum(),
      "| of these with a carrier date:", (delivered_no_delivery & orders["order_delivered_carrier_date"].notna()).sum())
print("canceled, has delivery date :", canceled_has_delivery.sum())

# Flag the two contradictions about delivery. Nothing is changed or dropped.
orders["status_conflict"] = delivered_no_delivery | canceled_has_delivery
print("status_conflict flag set on (delivered_no_delivery | canceled_has_delivery) :", orders["status_conflict"].sum())

delivered, no approval time : 14
delivered, no carrier date  : 2
delivered, no delivery date : 8 | of these with a carrier date: 7
canceled, has delivery date : 6
status_conflict flag set on (delivered_no_delivery | canceled_has_delivery) : 14


- Of the 8 delivered orders without a delivery date, 7 do have a carrier date, so only the last step is missing. 
- On the other side, 6 canceled orders have a full set of dates including delivery. 

**Decision:** keep all rows and set `status_conflict` on these 14 orders (8 + 6) without changing any value. They are 0.014% of orders and the true value cannot be proven. Filling the delivery date with the estimate would invent on-time deliveries. The 14 delivered orders without approval time and the 2 without carrier date stay empty. Carrier and delivery dates are not model inputs anyway.

#### Orders over time
#### By month
**Question:** how did the marketplace grow, and is every month usable?

In [9]:
purchase = orders["order_purchase_timestamp"]

# Reindex over the full month range so months with zero orders show up as zero
by_month = purchase.dt.to_period("M").value_counts().sort_index()
by_month = by_month.reindex(pd.period_range(by_month.index.min(), by_month.index.max(), freq="M"), fill_value=0)
display(by_month.rename("n_orders").to_frame())

,n_orders
2016-09,4
2016-10,324
2016-11,0
2016-12,1
2017-01,800
2017-02,1780
2017-03,2682
2017-04,2404
2017-05,3700
2017-06,3245


- The monthly table shows a young, fast-growing marketplace. 
- Volume starts at 800 orders in Jan 2017, climbs to a peak of 7,544 in Nov 2017 and then stays between about 6,000 and 7,300 per month until Aug 2018.
- The two ends of the data are thin. 
- 2016 has only 329 orders and Nov 2016 has none. 
- Sep and Oct 2018 have only 20 orders.

**NOTE**: This points to a modelling window of about Jan 2017 to Aug 2018 (to be decided in Part 2).

#### By weekday and time of day

In [10]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday = purchase.dt.day_name().value_counts().reindex(weekday_order).rename("n_orders").to_frame()
display(weekday)

# Group the 24 hours into six blocks of four hours
blocks = pd.cut(purchase.dt.hour, [-1, 3, 7, 11, 15, 19, 23],
                labels=["00-03", "04-07", "08-11", "12-15", "16-19", "20-23"])
blocks.value_counts().sort_index().rename("n_orders").to_frame()

,n_orders
order_purchase_timestamp,
Monday,16196
Tuesday,15963
Wednesday,15552
Thursday,14761
Friday,14122
Saturday,10887
Sunday,11960


,n_orders
order_purchase_timestamp,
00-03,4346
04-07,2127
08-11,20507
12-15,25536
16-19,24576
20-23,22349


- Monday is the busiest day (16,196 orders) and Saturday the quietest (10,887)
- So weekdays are busier than the weekend. 
- During the day the busiest block is 12:00 to 15:59 (25,536 orders) and the quietest is 04:00 to 07:59 (2,127 orders). 
- The time zone is not stated in the data.

#### Date logic checks
**Question:** do the dates follow the real order of events (purchase, approval, handoff, delivery)?

In [11]:
checks = {
    "approved before purchase":  orders["order_approved_at"] < orders["order_purchase_timestamp"],
    "carrier before approval":   orders["order_delivered_carrier_date"] < orders["order_approved_at"],
    "carrier before purchase":   orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"],
    "delivered before carrier":  orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"],
    "delivered before purchase": orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"],
    "estimate before purchase":  orders["order_estimated_delivery_date"] < orders["order_purchase_timestamp"],
}
display(pd.Series({k: int(v.sum()) for k, v in checks.items()}, name="n_orders").to_frame())

,n_orders
approved before purchase,0
carrier before approval,1359
carrier before purchase,166
delivered before carrier,23
delivered before purchase,0
estimate before purchase,0


Some checks are clean: 
- no order is approved, delivered or estimated before its purchase. 

Others are not: 
- 1,359 orders show a carrier handoff before the approval time.
- 166 of these orders even have the carrier date before the purchase, which is impossible. 
- A further 23 orders are delivered before the carrier handoff. The likely cause is late or wrong timestamps, but this cannot be checked from this file.

#### Durations and outliers
#### How long does each step take?
**Question:** how long are approval, handoff, transit, total delivery and the promise, and how late are orders?

In [12]:
DAY = 86400  # seconds per day

orders["approve_h"]  = (orders["order_approved_at"] - orders["order_purchase_timestamp"]).dt.total_seconds() / 3600
orders["handoff_d"]  = (orders["order_delivered_carrier_date"] - orders["order_approved_at"]).dt.total_seconds() / DAY
orders["transit_d"]  = (orders["order_delivered_customer_date"] - orders["order_delivered_carrier_date"]).dt.total_seconds() / DAY
orders["total_d"]    = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.total_seconds() / DAY
orders["promised_d"] = (orders["order_estimated_delivery_date"] - orders["order_purchase_timestamp"]).dt.total_seconds() / DAY

DUR_COLS = ["approve_h", "handoff_d", "transit_d", "total_d", "promised_d"]
orders[DUR_COLS].describe(percentiles=[0.05, 0.5, 0.95, 0.99]).T.round(2)

,count,mean,std,min,5%,50%,95%,99%,max
approve_h,99281.0,10.42,26.04,0.00,0.14,0.34,48.46,90.17,4509.18
handoff_d,97644.0,2.81,3.55,-171.22,0.25,1.82,8.10,17.15,125.76
transit_d,96475.0,9.33,8.76,-16.10,1.12,7.10,24.21,40.98,205.19
total_d,96476.0,12.56,9.55,0.53,3.02,10.22,29.28,46.05,209.63
promised_d,99441.0,23.77,8.83,1.65,10.47,23.24,38.48,50.42,155.14


- Approval is fast for most orders (median 0.34 hours) but has a long tail: the 99th percentile is 90 hours and the maximum is 4,509 hours (188 days). 
- From purchase to delivery (`total_d`) the median is 10.2 days, the 95th percentile 29.3 days and the maximum 209.6 days.
- The promised time (`promised_d`) has a median of 23.2 days, more than double the typical real time.